# Notebook 01 — OpenI Data Loading & QA Dataset Creation

This notebook:
1. Downloads the Indiana University CXR dataset via Kaggle (same data as OpenI)
2. Downloads the OpenI XML radiology reports
3. Generates a QA dataset using Groq LLaMA 3.1 8B Instant

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q groq python-dotenv tqdm pandas kaggle

In [ ]:
import os, sys
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/cxr_rag'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted at', DRIVE_ROOT)

In [ ]:
# ── Configure Kaggle credentials ─────────────────────────────────────────────
# Add KAGGLE_USERNAME and KAGGLE_KEY to Colab Secrets (key icon in left sidebar)
# Get them from: kaggle.com/settings -> API -> Create New Token

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# Verify
!kaggle datasets list --search 'indiana university chest' 2>&1 | head -5

In [ ]:
# ── Download images via Kaggle (~1 GB) ───────────────────────────────────────
# Same dataset as OpenI / Indiana University CXR
# Direct OpenI zip download is no longer supported by their website

IMAGES_DIR  = '/content/openi/images'
REPORTS_DIR = '/content/openi/reports'
REPORTS_TGZ = '/content/openi_reports.tgz'

import subprocess

if not os.path.exists(IMAGES_DIR):
    print('Downloading Indiana University CXR images via Kaggle (~1 GB) ...')
    os.makedirs('/content/openi', exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'raddar/chest-xrays-indiana-university',
        '-p', '/content/openi/',
        '--unzip'
    ], check=True)
    # The dataset extracts to /content/openi/images/
    print('Images downloaded.')
else:
    print('Images already exist.')

print('Contents of /content/openi/:', os.listdir('/content/openi/'))

In [ ]:
# ── Fix IMAGES_DIR path if Kaggle extracted differently ──────────────────────
# Kaggle dataset may extract images into a subfolder — auto-detect it
import glob

png_files = glob.glob('/content/openi/**/*.png', recursive=True)
if png_files:
    IMAGES_DIR = os.path.dirname(png_files[0])
    print(f'Found {len(png_files)} PNG images in: {IMAGES_DIR}')
else:
    print('WARNING: No PNG files found. Check /content/openi/ contents.')
    !find /content/openi/ -type f | head -20

In [ ]:
# ── Download OpenI XML reports (~20 MB) ──────────────────────────────────────
# Reports still available from OpenI directly

if not os.path.exists(REPORTS_DIR):
    print('Downloading OpenI reports (~20 MB) ...')
    subprocess.run([
        'wget', '-q',
        'https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz',
        '-O', REPORTS_TGZ
    ], check=True)
    os.makedirs(REPORTS_DIR, exist_ok=True)
    subprocess.run(['tar', '-xzf', REPORTS_TGZ, '-C', REPORTS_DIR], check=True)
    print('Reports downloaded.')
else:
    print('Reports already exist.')

print('Reports dir contents:', os.listdir(REPORTS_DIR)[:5])

In [ ]:
# ── Clone project repo ────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/mohamedtaha77/cxr-rag-system.git'

if not os.path.exists('/content/cxr-rag-system'):
    !git clone {REPO_URL} /content/cxr-rag-system
else:
    !git -C /content/cxr-rag-system pull

sys.path.insert(0, '/content/cxr-rag-system')
print('Repo ready.')

In [ ]:
# ── Parse OpenI XML reports ───────────────────────────────────────────────────
from src.data.openi_loader import OpenILoader

# Reports extract to ecgen-radiology/ subfolder
reports_subdir = os.path.join(REPORTS_DIR, 'ecgen-radiology')
if not os.path.exists(reports_subdir):
    reports_subdir = REPORTS_DIR

loader = OpenILoader(reports_dir=reports_subdir, images_dir=IMAGES_DIR)
df = loader.load()
print(f'Loaded {len(df)} studies with valid impression + frontal image')
df.head(3)

In [ ]:
# ── Train/val/test split and save ─────────────────────────────────────────────
import pandas as pd, shutil

train_df, val_df, test_df = loader.train_val_test_split(df)
full_df = pd.concat([train_df, val_df, test_df])

os.makedirs('/content/cxr-rag-system/data/processed', exist_ok=True)
full_df.to_csv('/content/cxr-rag-system/data/processed/reports_corpus.csv', index=False)
shutil.copy('/content/cxr-rag-system/data/processed/reports_corpus.csv',
            os.path.join(DRIVE_ROOT, 'reports_corpus.csv'))
print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

In [ ]:
# ── Configure Groq API ────────────────────────────────────────────────────────
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

from src.data.qa_creator import QACreator
creator = QACreator(groq_api_key=GROQ_API_KEY)
print('QA creator ready.')

In [ ]:
# ── Generate QA dataset ───────────────────────────────────────────────────────
# max_studies=500 → ~30 min on Groq free tier
# Set max_studies=None for full dataset (~3 hrs)

QA_OUTPUT = '/content/cxr-rag-system/data/processed/qa_dataset.jsonl'

pairs = creator.generate_dataset(
    df=full_df,
    output_path=QA_OUTPUT,
    max_studies=500,
)

print(f'\nGenerated {len(pairs)} QA pairs')
shutil.copy(QA_OUTPUT, os.path.join(DRIVE_ROOT, 'qa_dataset.jsonl'))

In [ ]:
# ── Inspect sample QA pairs ───────────────────────────────────────────────────
import json
with open(QA_OUTPUT) as f:
    samples = [json.loads(l) for l in f][:5]

for s in samples:
    print(f"Category : {s['category']}")
    print(f"Q        : {s['question']}")
    print(f"A        : {s['answer']}")
    print()

In [ ]:
# ── Dataset statistics ────────────────────────────────────────────────────────
qa_df = pd.read_json(QA_OUTPUT, lines=True)
print('Category distribution:')
print(qa_df['category'].value_counts())
print(f'\nTotal pairs : {len(qa_df)}')
print(f'Unique studies: {qa_df["study_id"].nunique()}')
print(f'Split dist.:\n{qa_df["split"].value_counts()}')